# Генеративно-состязательные сети (GAN) — интерактивный конспект

Конспект по теме домашнего задания **«23.1 Генеративно-состязательные модели»**.

В исходном задании обучается DCGAN, генерирующий лица людей (датасет FFHQ, ~3000
изображений, 35 эпох). Здесь мы **не скачиваем 4 ГБ данных и ничего не обучаем
всерьёз** — вместо этого на маленьких синтетических тензорах разбираем устройство
каждого компонента GAN:

- генератор и дискриминатор, состязательная minimax-игра;
- блоки `ConvTranspose2d` (генератор) и `Conv2d`-stride (дискриминатор);
- расчёт BCE-лосса для шага дискриминатора и шага генератора;
- label smoothing;
- мини-GAN на синтетических 1D-данных (обучается за пару секунд);
- идея leave-one-out 1-NN и t-SNE на крошечном примере.

Каждая ячейка считается за доли секунды.

**Источники:** [Goodfellow et al., 2014](https://arxiv.org/abs/1406.2661) (GAN),
[Radford et al., 2015](https://arxiv.org/abs/1511.06434) (DCGAN),
[PyTorch DCGAN Tutorial](https://pytorch.org/tutorials/beginner/dcgan_faces_tutorial.html).


## 0. Импорты и настройка

Понадобятся `torch`, `numpy`, `matplotlib` и пара функций из `scikit-learn`.
Фиксируем сиды (`SEED = 42`) для воспроизводимости.


In [ ]:
import random

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import accuracy_score
from sklearn.manifold import TSNE

# Фиксируем сиды — воспроизводимость (random_state=42 по конвенции проекта).
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# В конспекте всё считаем на CPU — выборки крошечные, GPU не нужен.
device = torch.device('cpu')
print('PyTorch:', torch.__version__, '| устройство:', device)

## 1. Идея GAN: две сети-антагониста

**GAN (Generative Adversarial Network)** состоит из двух нейросетей, обучающихся
одновременно и *соревнуясь*:

- **Генератор `G`** превращает случайный шум `z` в объект (изображение). Цель —
  выдавать настолько правдоподобные картинки, чтобы дискриминатор не отличил их
  от настоящих.
- **Дискриминатор `D`** — бинарный классификатор: получает изображение и выдаёт
  вероятность того, что оно настоящее (метка 1), а не сгенерированное (метка 0).

Это игра двух игроков с нулевой суммой — **minimax**:

$$\min_G \max_D \; \mathbb{E}_{x \sim p_{data}}[\log D(x)] +
\mathbb{E}_{z \sim p_z}[\log(1 - D(G(z)))]$$

`D` хочет **максимизировать** функционал (точно ловить фейки), `G` — **минимизировать**
(заставить `D` ошибаться). В идеале `G` воспроизводит распределение данных, а `D`
для любого входа выдаёт 0.5 (не может отличить реальное от фейка).

Схематически поток данных:

```
   z ~ N(0,1)  ──►  Генератор G  ──►  fake  ──┐
                                              ├──►  Дискриминатор D  ──►  p ∈ [0,1]
   реальные данные x  ───────────────────────┘
```


## 2. Блок генератора: `ConvTranspose2d`

Генератор должен из вектора шума `latent_size x 1 x 1` «вырастить» полноразмерную
картинку `3 x H x W`. Для этого используется **транспонированная свёртка**
`nn.ConvTranspose2d` — обучаемый апсемплинг (увеличение пространственного размера).

Типовой DCGAN-блок генератора: `ConvTranspose2d -> BatchNorm2d -> ReLU`.

- `ConvTranspose2d(in, out, kernel_size=4, stride=2, padding=1)` **удваивает**
  высоту и ширину;
- первый блок генератора с `stride=1, padding=0`, ядро 4x4 переводит `1x1 -> 4x4`;
- `BatchNorm` стабилизирует обучение, `ReLU` — нелинейность.

Посмотрим, как меняется форма тензора.


In [ ]:
# Один блок генератора: ConvTranspose2d -> BatchNorm -> ReLU.
gen_block = nn.Sequential(
    nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1, bias=False),
    nn.BatchNorm2d(64),
    nn.ReLU(inplace=True),
)

x = torch.randn(8, 128, 16, 16)   # батч 8, 128 каналов, размер 16x16
y = gen_block(x)
print('вход :', tuple(x.shape))
print('выход:', tuple(y.shape), '<- размер удвоился (16 -> 32)')

# Первый блок генератора: из латентного вектора 1x1 делает карту 4x4.
first_block = nn.ConvTranspose2d(128, 512, kernel_size=4, stride=1, padding=0)
z = torch.randn(8, 128, 1, 1)     # латентный вектор как "картинка" 1x1
print()
print('латентный вектор:', tuple(z.shape))
print('после 1-го блока:', tuple(first_block(z).shape), '<- 1x1 -> 4x4')

**Мини-генератор целиком.** Соберём цепочку блоков, которая из латентного
вектора `64 x 1 x 1` делает картинку `3 x 32 x 32`. На выходе — `Tanh`, который
зажимает значения в диапазон `[-1, 1]`.


In [ ]:
latent_size = 64

# Мини-генератор: 1x1 -> 4 -> 8 -> 16 -> 32. Каналы убывают, размер растёт.
mini_generator = nn.Sequential(
    nn.ConvTranspose2d(latent_size, 128, 4, stride=1, padding=0, bias=False),
    nn.BatchNorm2d(128), nn.ReLU(inplace=True),                      # -> 4x4
    nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1, bias=False),
    nn.BatchNorm2d(64), nn.ReLU(inplace=True),                       # -> 8x8
    nn.ConvTranspose2d(64, 32, 4, stride=2, padding=1, bias=False),
    nn.BatchNorm2d(32), nn.ReLU(inplace=True),                       # -> 16x16
    nn.ConvTranspose2d(32, 3, 4, stride=2, padding=1, bias=False),
    nn.Tanh(),                                                       # -> 32x32
)

noise = torch.randn(4, latent_size, 1, 1)
fake = mini_generator(noise)
print('шум на входе   :', tuple(noise.shape))
print('картинка на выходе:', tuple(fake.shape))
print('диапазон значений: [{:.3f}, {:.3f}] (Tanh -> [-1, 1])'.format(
    fake.min().item(), fake.max().item()))

## 3. Блок дискриминатора: `Conv2d` со `stride`

Дискриминатор делает обратное генератору: из картинки `3 x H x W` постепенно
получает один скаляр-вероятность. Для уменьшения размера используется обычная
свёртка `nn.Conv2d` со `stride=2` (обучаемый даунсемплинг — замена pooling).

Типовой DCGAN-блок дискриминатора: `Conv2d -> BatchNorm2d -> LeakyReLU(0.2)`.

- `Conv2d(in, out, kernel_size=4, stride=2, padding=1)` **вдвое уменьшает** размер;
- `LeakyReLU(0.2)` вместо `ReLU`: при отрицательном входе градиент не зануляется
  полностью (`f(x) = 0.2x` при `x < 0`) — важно, чтобы сигнал доходил до генератора;
- финальная свёртка сжимает карту в `1x1`, `Sigmoid` переводит логит в вероятность.


In [ ]:
# Один блок дискриминатора: Conv2d(stride=2) -> BatchNorm -> LeakyReLU.
disc_block = nn.Sequential(
    nn.Conv2d(3, 64, kernel_size=4, stride=2, padding=1, bias=False),
    nn.BatchNorm2d(64),
    nn.LeakyReLU(0.2, inplace=True),
)

img = torch.randn(8, 3, 32, 32)
out = disc_block(img)
print('вход :', tuple(img.shape))
print('выход:', tuple(out.shape), '<- размер уполовинен (32 -> 16)')

**Мини-дискриминатор целиком.** Соберём цепочку, которая из картинки
`3 x 32 x 32` выдаёт один скаляр в `[0, 1]` — вероятность «реальности».
`Flatten` убирает лишние оси, чтобы выход был формы `[B]`.


In [ ]:
# Мини-дискриминатор: 32 -> 16 -> 8 -> 4 -> 1. Каналы растут, размер убывает.
mini_discriminator = nn.Sequential(
    nn.Conv2d(3, 32, 4, stride=2, padding=1, bias=False),
    nn.LeakyReLU(0.2, inplace=True),                                 # -> 16x16
    nn.Conv2d(32, 64, 4, stride=2, padding=1, bias=False),
    nn.BatchNorm2d(64), nn.LeakyReLU(0.2, inplace=True),             # -> 8x8
    nn.Conv2d(64, 128, 4, stride=2, padding=1, bias=False),
    nn.BatchNorm2d(128), nn.LeakyReLU(0.2, inplace=True),            # -> 4x4
    nn.Conv2d(128, 1, 4, stride=1, padding=0, bias=False),
    nn.Sigmoid(),                                                    # -> 1x1
    nn.Flatten(start_dim=0),                                         # -> [B]
)

img_batch = torch.randn(4, 3, 32, 32)
prob = mini_discriminator(img_batch)
print('картинки на входе :', tuple(img_batch.shape))
print('вероятности выхода:', tuple(prob.shape), '<- по одному числу на картинку')
print('значения:', np.round(prob.detach().numpy(), 3), '(все в [0, 1])')

## 4. DCGAN-инициализация весов

DCGAN рекомендует особую инициализацию весов — без неё GAN заводится менее
стабильно:

- свёртки (`Conv2d`, `ConvTranspose2d`) -> `N(0, 0.02)`;
- веса `BatchNorm` -> `N(1, 0.02)`, bias -> нули.

Применяется ко всей сети одним вызовом `model.apply(weights_init)`.


In [ ]:
def weights_init(m):
    """Инициализация по схеме DCGAN: Conv -> N(0, 0.02); BatchNorm -> N(1, 0.02)."""
    name = m.__class__.__name__
    if name.find('Conv') != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif name.find('BatchNorm') != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0.0)


mini_generator.apply(weights_init)
mini_discriminator.apply(weights_init)

# Проверим: std весов первой свёртки генератора должно быть около 0.02.
w = mini_generator[0].weight.data
print('std весов первой ConvTranspose2d: {:.4f} (ожидаем ~0.02)'.format(w.std().item()))
print('mean весов:                       {:.4f} (ожидаем ~0.0)'.format(w.mean().item()))

## 5. BCE-лосс: шаг дискриминатора и шаг генератора

Лосс GAN — **бинарная кросс-энтропия** (`nn.BCELoss`), применяется к выходу после
`Sigmoid`:

$$\text{BCE}(p, y) = -[\, y \log p + (1 - y) \log(1 - p)\,]$$

И генератор, и дискриминатор используют один и тот же BCE — **различаются только
целевые метки `y`**, которые им подставляются.

### Шаг дискриминатора
- реальные изображения -> целевая метка **1** (с label smoothing — **0.9**);
- фейковые изображения -> целевая метка **0**;
- `loss_D = BCE(D(real), 0.9) + BCE(D(fake.detach()), 0.0)`.

`.detach()` критичен: при шаге `D` градиент не должен течь в генератор.

### Шаг генератора
- фейковые изображения -> целевая метка **1** (генератор хочет, чтобы `D` принял
  их за реальные);
- `loss_G = BCE(D(fake), 1.0)`.

Подстановка метки 1 фейкам — это **non-saturating loss**: генератор максимизирует
`log D(G(z))`, что даёт сильный градиент даже когда дискриминатор уверен (борьба
с vanishing gradients).


In [ ]:
bce = nn.BCELoss()
batch = 16

# Симулируем выход дискриминатора (вероятности после Sigmoid).
# "хороший" дискриминатор: на реальных выдаёт ~высокие p, на фейках ~низкие.
d_on_real = torch.rand(batch) * 0.3 + 0.65   # ~0.65..0.95
d_on_fake = torch.rand(batch) * 0.3 + 0.05   # ~0.05..0.35

# --- Шаг дискриминатора ---
real_targets = torch.full((batch,), 0.9)     # label smoothing: 1 -> 0.9
fake_targets = torch.zeros(batch)            # фейки -> 0
loss_d = bce(d_on_real, real_targets) + bce(d_on_fake, fake_targets)
print('loss_D = BCE(D(real), 0.9) + BCE(D(fake), 0.0) = {:.4f}'.format(loss_d.item()))

# --- Шаг генератора ---
# Те же фейки, но целевая метка = 1: генератор хочет обмануть дискриминатор.
gen_targets = torch.ones(batch)
loss_g = bce(d_on_fake, gen_targets)
print('loss_G = BCE(D(fake), 1.0)                    = {:.4f}'.format(loss_g.item()))
print()
print('Заметьте: loss_G заметно выше — генератору пока не удаётся обмануть D.')

### Зачем нужен label smoothing — наглядно

Если дискриминатор стал «слишком уверен» (выдаёт на реальных `p` близко к 1.0),
обычная метка `1.0` почти не даёт ему лосса и градиента. Сглаживание до `0.9`
оставляет небольшой штраф за чрезмерную уверенность — это стабилизирует обучение
и не даёт дискриминатору «задавить» генератор.


In [ ]:
# Дискриминатор очень уверен: на реальных выдаёт p близко к 1.
d_very_confident = torch.full((8,), 0.99)

loss_hard = bce(d_very_confident, torch.ones(8))           # метка 1.0
loss_smooth = bce(d_very_confident, torch.full((8,), 0.9)) # метка 0.9

print('BCE с меткой 1.0  (без сглаживания): {:.4f}'.format(loss_hard.item()))
print('BCE с меткой 0.9  (label smoothing): {:.4f}'.format(loss_smooth.item()))
print()
print('Со сглаживанием лосс не схлопывается в почти-ноль -> у D остаётся')
print('градиент, он не становится переуверенным.')

## 6. Мини-GAN на синтетических данных (обучение за пару секунд)

Соберём **полноценный, но крошечный GAN** на 1D-данных, чтобы увидеть алгоритм
обучения целиком и убедиться, что лоссы GAN **не сходятся к нулю**, а колеблются.

**Задача-игрушка.** «Реальные данные» — числа из нормального распределения
`N(4, 0.5)`. Генератор получает шум `N(0,1)` и должен научиться выдавать числа,
похожие на реальные. Дискриминатор отличает реальные числа от сгенерированных.
Сети — крошечные `Linear`-MLP, обучение занимает пару секунд на CPU.


In [ ]:
# --- "Реальное" распределение данных: N(4, 0.5) ---
REAL_MEAN, REAL_STD = 4.0, 0.5


def sample_real(n):
    """Батч 'реальных' чисел формы [n, 1]."""
    return torch.randn(n, 1) * REAL_STD + REAL_MEAN


# --- Крошечные генератор и дискриминатор (MLP) ---
G = nn.Sequential(
    nn.Linear(1, 16), nn.ReLU(),
    nn.Linear(16, 1),                    # шум [n,1] -> число [n,1]
)
D = nn.Sequential(
    nn.Linear(1, 16), nn.LeakyReLU(0.2),
    nn.Linear(16, 1), nn.Sigmoid(),      # число [n,1] -> вероятность [n,1]
)

print('Генератор :', sum(p.numel() for p in G.parameters()), 'параметров')
print('Дискриминатор:', sum(p.numel() for p in D.parameters()), 'параметров')

In [ ]:
# --- Обучение мини-GAN ---
bce = nn.BCELoss()
opt_g = torch.optim.Adam(G.parameters(), lr=0.01, betas=(0.5, 0.999))
opt_d = torch.optim.Adam(D.parameters(), lr=0.01, betas=(0.5, 0.999))

EPOCHS, BATCH = 400, 128
d_hist, g_hist = [], []

for epoch in range(EPOCHS):
    real = sample_real(BATCH)
    noise = torch.randn(BATCH, 1)

    # ---------- Шаг дискриминатора ----------
    opt_d.zero_grad()
    fake = G(noise)
    # реальные -> 0.9 (label smoothing), фейки -> 0; .detach() -> градиент не в G
    loss_d = (bce(D(real), torch.full((BATCH, 1), 0.9)) +
              bce(D(fake.detach()), torch.zeros(BATCH, 1)))
    loss_d.backward()
    opt_d.step()

    # ---------- Шаг генератора ----------
    opt_g.zero_grad()
    # те же фейки, целевая метка 1: генератор хочет обмануть дискриминатор
    loss_g = bce(D(fake), torch.ones(BATCH, 1))
    loss_g.backward()
    opt_g.step()

    d_hist.append(loss_d.item())
    g_hist.append(loss_g.item())

print('Обучение завершено за {} эпох.'.format(EPOCHS))
print('Финальные лоссы: loss_D = {:.4f}, loss_G = {:.4f}'.format(
    d_hist[-1], g_hist[-1]))

**Проверим результат.** Если GAN обучился, генератор выдаёт числа со средним
около `4.0` (как у реальных данных). Сравним статистику и посмотрим на графики
лоссов и распределений.


In [ ]:
# Сгенерируем числа обученным генератором и сравним с реальными.
with torch.no_grad():
    gen_samples = G(torch.randn(2000, 1)).squeeze().numpy()
real_samples = sample_real(2000).squeeze().numpy()

print('Реальные данные      : mean = {:.3f}, std = {:.3f}'.format(
    real_samples.mean(), real_samples.std()))
print('Сгенерированные данные: mean = {:.3f}, std = {:.3f}'.format(
    gen_samples.mean(), gen_samples.std()))
print('(цель генератора: mean ~ {:.1f})'.format(REAL_MEAN))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

# График лоссов: видно, что они колеблются, а НЕ сходятся к нулю.
ax1.plot(d_hist, label='loss_D (дискриминатор)', alpha=0.8)
ax1.plot(g_hist, label='loss_G (генератор)', alpha=0.8)
ax1.set_xlabel('Эпоха'); ax1.set_ylabel('BCE-лосс')
ax1.set_title('Лоссы GAN колеблются, не сходятся к 0')
ax1.legend()

# Гистограммы: распределение генератора подтянулось к реальному.
ax2.hist(real_samples, bins=40, alpha=0.6, label='Реальные', color='tab:blue')
ax2.hist(gen_samples, bins=40, alpha=0.6, label='Сгенерированные', color='tab:red')
ax2.set_xlabel('Значение'); ax2.set_ylabel('Частота')
ax2.set_title('Распределения после обучения')
ax2.legend()

plt.tight_layout()
plt.show()

**Что видно на графиках.**

- **Лоссы не сходятся к нулю**, а колеблются вокруг некоторого уровня — это норма
  для GAN. Генератор и дискриминатор — антагонисты: улучшение одного ухудшает
  лосс другого, поэтому равновесие выглядит как колебание, а не монотонное падение.
- **Гистограммы** реальных и сгенерированных чисел заметно перекрываются —
  генератор научился воспроизводить распределение `N(4, 0.5)`, хотя видел только
  сигнал от дискриминатора и ни разу — реальные данные напрямую.

В исходном задании всё устроено так же, только данные — изображения лиц 3x128x128,
а сети — глубокие свёрточные DCGAN.


## 7. Оценка качества: leave-one-out 1-NN accuracy

Смотреть на сгенерированные картинки глазами субъективно. Количественная
альтернатива — **leave-one-out 1-NN accuracy**:

1. Берём `N` реальных и `N` сгенерированных объектов; реальным метка **1**,
   фейковым — **0**.
2. По схеме **leave-one-out** обучаем `KNeighborsClassifier(n_neighbors=1)` на
   всех объектах, кроме одного, и проверяем предсказание класса на отложенном.
3. Так перебираем все объекты, считаем итоговую accuracy.

**Как читать метрику:**

- **accuracy ~ 0.5 — идеал.** Фейки неотличимы от реальных: ближайший сосед
  любого объекта с равной вероятностью реальный или фейковый, 1-NN угадывает
  на уровне монетки.
- **accuracy -> 1.0 — плохо.** Чем легче 1-NN отделяет фейки, тем дальше выборки
  генератора лежат от реальных.

Покажем оба крайних случая на 1D-данных из мини-GAN.


In [ ]:
def loo_1nn_accuracy(real_1d, fake_1d):
    """Leave-one-out 1-NN accuracy для двух наборов 1D-чисел."""
    X = np.concatenate([real_1d, fake_1d]).reshape(-1, 1)
    y = np.concatenate([np.ones(len(real_1d), dtype=int),
                        np.zeros(len(fake_1d), dtype=int)])
    loo = LeaveOneOut()
    pred = np.zeros(len(y), dtype=int)
    for tr, te in loo.split(X):
        knn = KNeighborsClassifier(n_neighbors=1).fit(X[tr], y[tr])
        pred[te] = knn.predict(X[te])
    return accuracy_score(y, pred)


# Берём по 150 объектов (LOO имеет сложность O(N^2) -> выборку держим небольшой).
real = sample_real(150).squeeze().numpy()

# Случай A: ХОРОШИЙ генератор — фейки из того же распределения N(4, 0.5).
good_fake = (torch.randn(150) * REAL_STD + REAL_MEAN).numpy()
acc_good = loo_1nn_accuracy(real, good_fake)

# Случай B: ПЛОХОЙ генератор — фейки из совсем другого распределения N(0, 1).
bad_fake = torch.randn(150).numpy()
acc_bad = loo_1nn_accuracy(real, bad_fake)

print('Хороший генератор (фейки ~ реальные): LOO 1-NN accuracy = {:.3f}'.format(acc_good))
print('   -> близко к 0.5: 1-NN не может отличить фейк от реального (это идеал)')
print()
print('Плохой генератор (фейки из N(0,1))   : LOO 1-NN accuracy = {:.3f}'.format(acc_bad))
print('   -> близко к 1.0: распределения не пересекаются, фейки легко отличимы')

В прогоне исходного решения leave-one-out 1-NN accuracy получилась **0.8942**
(по 600 реальных и 600 сгенерированных лиц, уменьшенных до 32x32). Значение
заметно выше идеальных `0.5` — DCGAN, обученный 35 эпох на ~3000 лицах, ещё
довольно легко отличим. Это ожидаемо: задание оценивает корректность реализации,
а не фотореализм.


## 8. Оценка качества: t-SNE визуализация распределений

[**t-SNE**](https://scikit-learn.org/stable/modules/generated/sklearn.manifold.TSNE.html)
снижает размерность многомерных векторов до 2D, сохраняя локальную структуру
(близкие объекты остаются близкими). Берём flatten-векторы реальных и
сгенерированных изображений, проецируем в 2D и рисуем scatter, раскрашивая точки
по происхождению.

**Как читать график:** чем сильнее **перемешаны** точки реальных и
сгенерированных объектов — тем ближе распределение генератора к реальному. Два
**обособленных** облака означают, что фейки систематически отличаются от реальных.
t-SNE — качественная иллюстрация к метрике из раздела 7.

Покажем на синтетических многомерных данных оба случая.


In [ ]:
# Синтетика: "реальные" объекты — точки в 50-мерном пространстве вокруг центра A.
DIM = 50
real_vecs = np.random.randn(120, DIM) + 3.0      # облако вокруг (3,3,...,3)

# Случай A: хороший генератор — фейки из того же облака, что и реальные.
fake_good = np.random.randn(120, DIM) + 3.0

# Случай B: плохой генератор — фейки смещены в сторону (центр около 0).
fake_bad = np.random.randn(120, DIM)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, fake, title in [(axes[0], fake_good, 'Хороший генератор: облака перемешаны'),
                        (axes[1], fake_bad, 'Плохой генератор: облака разделены')]:
    X = np.concatenate([real_vecs, fake])
    # perplexity должен быть меньше числа объектов; для 240 точек 30 подходит.
    emb = TSNE(n_components=2, random_state=42, perplexity=30,
               init='pca').fit_transform(X)
    n = len(real_vecs)
    ax.scatter(emb[:n, 0], emb[:n, 1], c='tab:blue', s=20, alpha=0.6,
               label='Реальные')
    ax.scatter(emb[n:, 0], emb[n:, 1], c='tab:red', s=20, alpha=0.6,
               label='Сгенерированные')
    ax.set_title(title)
    ax.set_xlabel('t-SNE 1'); ax.set_ylabel('t-SNE 2')
    ax.legend()

plt.tight_layout()
plt.show()
print('Слева облака сливаются -> распределения близки -> LOO 1-NN accuracy ~0.5.')
print('Справа облака раздельны -> распределения различны -> accuracy ~1.0.')

В исходном решении t-SNE дал **два заметно разделённых облака** — это
согласуется с высокой leave-one-out accuracy `0.8942`: распределения реальных и
сгенерированных лиц пока различаются. Метрика 7 и визуализация 8 — согласованный
сигнал, что простому DCGAN ещё есть куда расти.


## 9. Попробуй сам

Небольшие задания для закрепления. Меняйте код в ячейках ниже и смотрите на
результат.

### Задание 1. Форма выхода `ConvTranspose2d`

Подберите `kernel_size`, `stride`, `padding` так, чтобы блок `ConvTranspose2d`
превратил карту `8x8` в `16x16`.

*Подсказка:* для удвоения размера в DCGAN используют `kernel_size=4, stride=2,
padding=1`.


In [ ]:
# TODO: подберите параметры так, чтобы получить выход 16x16.
# block = nn.ConvTranspose2d(32, 16, kernel_size=?, stride=?, padding=?)
# x = torch.randn(4, 32, 8, 8)
# print(tuple(block(x).shape))   # цель: (4, 16, 16, 16)

# --- Решение (раскомментируйте для проверки) ---
# block = nn.ConvTranspose2d(32, 16, kernel_size=4, stride=2, padding=1)
# x = torch.randn(4, 32, 8, 8)
# print(tuple(block(x).shape))

### Задание 2. Влияние label smoothing

Посчитайте BCE-лосс для очень уверенного дискриминатора (`p = 0.999`) с обычной
меткой `1.0` и со сглаженной `0.8`. Какой из лоссов больше и почему сглаживание
помогает обучению?

*Подсказка:* при метке `1.0` лосс уверенного `D` почти ноль -> почти нет
градиента; сглаживание оставляет штраф за переуверенность.


In [ ]:
# TODO: сравните BCE-лосс с меткой 1.0 и со сглаженной меткой 0.8.
# bce = nn.BCELoss()
# p = torch.full((8,), 0.999)
# loss_hard = bce(p, ...)     # метка 1.0
# loss_soft = bce(p, ...)     # метка 0.8
# print(loss_hard.item(), loss_soft.item())

# --- Решение (раскомментируйте для проверки) ---
# bce = nn.BCELoss()
# p = torch.full((8,), 0.999)
# print('метка 1.0:', bce(p, torch.ones(8)).item())
# print('метка 0.8:', bce(p, torch.full((8,), 0.8)).item())

### Задание 3. Эксперимент с мини-GAN

Измените `REAL_MEAN` (например, на `-2.0`) и переобучите мини-GAN из раздела 6.
Убедитесь, что генератор подстраивается под новое распределение — `mean`
сгенерированных чисел станет около нового `REAL_MEAN`.

*Подсказка:* достаточно поменять `REAL_MEAN` и заново выполнить ячейки обучения
и проверки из раздела 6.

### Задание 4. Зависимость 1-NN accuracy от качества генератора

В функции `loo_1nn_accuracy` из раздела 7 постепенно сближайте распределение
«плохих» фейков с реальным (`bad_fake` смещайте от `N(0,1)` к `N(4, 0.5)`) и
проследите, как accuracy падает от ~1.0 к ~0.5.

*Подсказка:* `fake = torch.randn(150) * 0.5 + shift`, перебирайте
`shift` в диапазоне от 0 до 4.


In [ ]:
# TODO: проследите, как LOO 1-NN accuracy падает к 0.5 по мере сближения
#        распределений фейков и реальных данных.
# real = sample_real(150).squeeze().numpy()
# for shift in [0.0, 1.0, 2.0, 3.0, 4.0]:
#     fake = (torch.randn(150) * 0.5 + shift).numpy()
#     print(f'shift={shift}: accuracy = {loo_1nn_accuracy(real, fake):.3f}')

# --- Решение (раскомментируйте для проверки) ---
# real = sample_real(150).squeeze().numpy()
# for shift in [0.0, 1.0, 2.0, 3.0, 4.0]:
#     fake = (torch.randn(150) * 0.5 + shift).numpy()
#     print(f'shift={shift}: accuracy = {loo_1nn_accuracy(real, fake):.3f}')

## 10. Итоги

- **GAN** = две сети-антагониста: **генератор** делает фейки из шума,
  **дискриминатор** отличает фейки от реальных. Обучение — minimax-игра.
- **Лоссы GAN не сходятся к нулю.** Здоровое обучение — колебание обоих лоссов
  вокруг равновесия; падение лосса к нулю означает, что один из игроков «победил»
  (плохой признак).
- **DCGAN-рецепты:** `ConvTranspose2d` для апсемплинга в генераторе, `Conv2d` со
  `stride` для даунсемплинга в дискриминаторе, `BatchNorm`, `ReLU` в `G` /
  `LeakyReLU(0.2)` в `D`, `Tanh` на выходе генератора, инициализация весов
  `N(0, 0.02)`.
- **Лосс — BCE**, одинаковый для обеих сетей; различаются только целевые метки:
  `D` учит реальные->0.9 (label smoothing) и фейки->0; `G` подставляет фейкам
  метку 1 (non-saturating loss — борьба с vanishing gradients).
- **`.detach()`** при шаге дискриминатора обязателен — иначе градиент потечёт в
  генератор. Генератор и дискриминатор обновляются раздельными оптимизаторами.
- **Оценка качества:** leave-one-out 1-NN accuracy (количественно, идеал ~0.5) и
  t-SNE (качественно — перекрытие облаков реальных и сгенерированных объектов).
- **Результаты прогона исходного решения:** 3143 изображения FFHQ, 35 эпох,
  финальные `loss_D = 0.6044`, `loss_G = 5.2999` (без коллапса), LOO 1-NN
  accuracy = **0.8942** (выше идеала 0.5 — простой DCGAN ещё отличим, что
  ожидаемо).

**Что почитать дальше:**
[GAN, Goodfellow 2014](https://arxiv.org/abs/1406.2661) ·
[DCGAN, Radford 2015](https://arxiv.org/abs/1511.06434) ·
[Improved Techniques for Training GANs, Salimans 2016](https://arxiv.org/abs/1606.03498) ·
[PyTorch DCGAN Tutorial](https://pytorch.org/tutorials/beginner/dcgan_faces_tutorial.html).
